In [ ]:
!pip install pyspark --quiet
print("Pyspark installation complete")


Pyspark installation complete


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
spark=SparkSession.builder.appName('Day4 BigData sale').config('spark.sql.adaptive.enabled','true').getOrCreate()
print(f'Spark version : {spark.version}')
print(f'SparkSession :Active')

Spark version : 4.0.2
SparkSession :Active


In [ ]:
df_bronze = spark.read \
       .option('header', 'true') \
       .option('inferSchema', 'true') \
       .csv('large_sales_data.csv')
print('=== BRONZE LAYER - Raw Data ===')
print(f'Rows : {df_bronze.count()}')
print(f'Columns : {len(df_bronze.columns)}')
print(f'Names : {df_bronze.columns}')
print()
df_bronze.printSchema()

=== BRONZE LAYER - Raw Data ===
Rows : 5000
Columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
#Inspect first rows and summary statistics
print("First 5 rows:")
df_bronze.show(5,truncate = False)
print("\nBasics statistics for numeric columns:")
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [ ]:
df_bronze.write \
.mode('overwrite') \
.parquet('sales_bronze.parquet')
print('Bronze Parquet saved: sales_bronze.parquet')

import os
def get_dir_size(path):
  """Get size of a file or directory in KB"""
  if os.path.isfile(path):
    return os.path.getsize(path) / 1024
  total=0
  for dirpath,dirnames,filenames in os.walk(path):
    for f in filenames:
      total +=os.path.getsize(os.path.join(dirpath,f))
  return  total / 1024
csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction=(1 - parquet_size/csv_size) * 100
print(f'CSV file size: {csv_size:.2f} KB')
print(f'Parquet file size: {parquet_size:.2f} KB')
print(f'Reduction in size: {reduction:.1f}% smaller')
print(f'At 1 TB scale:CSV=1000GB -> Parquet={1000*(1-reduction/100):0f} GB')

Bronze Parquet saved: sales_bronze.parquet
CSV file size: 529.31 KB
Parquet file size: 55.10 KB
Reduction in size: 89.6% smaller
At 1 TB scale:CSV=1000GB -> Parquet=104.092868 GB


In [ ]:
df_silver=df_bronze\
.dropDuplicates()\
.dropna(subset=['order_id','product','revenue'])
df_silver=df_silver.withColumn(
    'order_date',
    to_date(col('order_date'),'dd-MM-yyyy')
)
df_silver=df_silver\
.withColumn('order_year',year(col('order_date')))\
.withColumn('order_month',month(col('order_date')))
df_silver=df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue')>40000,'High')
    .when(col('revenue')>10000,'Medium')
    .otherwise('Low')
)
print(f"Silver layer rows:{df_silver.count()}")
print("New columns added:order_year,order_month,revenue_category")
df_silver.select('product','revenue','order_year','order_month','revenue_category').show()

Silver layer rows:5000
New columns added:order_year,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|  Keyboard|  13200|      2023|          2|          Medium|
|    Webcam|  17500|      2023|          1|          Medium|
|   Speaker|  58500|      2023|          4|            High|
|  Keyboard|   9600|      2023|         12|             Low|
|    Laptop| 180000|      2023|          8|            High|
|Headphones|  38500|      2023|          5|          Medium|
|    Webcam|  35000|      2023|         11|          Medium|
|    Laptop| 360000|      2023|          1|            High|
|    Tablet| 320000|      2023|          6|            High|
|    Laptop| 225000|      2023|          6|            High|
|     Mouse|   6400|      2023|          8|             Low|
|   Monitor| 132000|      2023|          7|            High|
|   